# Digital Industry Analytics Platform — Exploratory Analysis

This notebook demonstrates the core analytical workflow used by the platform:
load → clean → enrich → correlate → regress → forecast.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import load_primary_panel
from src.data_cleaning import clean_dataset, generate_quality_report
from src.indicator_engine import enrich_indicators, ranking_table
from src.analytics import correlation_matrix, linear_regression_analysis, descriptive_statistics
from src.forecasting import run_forecast

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 50)

## 1. Load & data quality

In [ ]:
raw = load_primary_panel()
report = generate_quality_report(raw, source="digital_industry_indicators.csv")
print(report.to_json())
raw.head()

## 2. Clean & enrich indicators

In [ ]:
clean, _ = clean_dataset(raw, persist_report=False)
df = enrich_indicators(clean)
print(df.shape)
df.describe()

## 3. Correlation & regression

In [ ]:
cols = [
    "digital_index",
    "technology_investment",
    "revenue_growth",
    "rd_ratio",
    "digital_development_index",
]
corr = correlation_matrix(df, columns=cols)

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(corr, annot=True, cmap="Teal", vmin=-1, vmax=1, ax=ax)
ax.set_title("Indicator correlations")
plt.tight_layout()
plt.show()

reg = linear_regression_analysis(df)
reg.to_dict()

## 4. Rankings & forecast

In [ ]:
display(ranking_table(df, metric="digital_development_index", group_by="industry"))

fc = run_forecast(df, metric="digital_index", method="ensemble", horizon=5)
print(fc.method, fc.metrics)
fc.forecast